### Multi conditional routing

In [0]:
!pip install langchain_core databricks-langchain langgraph
dbutils.library.restartPython()

In [0]:
import operator
from databricks_langchain import ChatDatabricks
from typing import TypedDict, Annotated, Literal, List
from typing_extensions import TypedDict, Literal
from langgraph.graph import StateGraph, START, END
from langgraph.graph import MessagesState
from langchain_core.prompts import ChatPromptTemplate
from langgraph.types import Send
from langchain.tools import tool
from langchain.messages import HumanMessage, SystemMessage, ToolMessage
from pydantic import BaseModel, Field
from IPython.display import Image, display

### Loan example

In [0]:
import random
import operator
import pandas as pd
import numpy as np
import os
from typing import TypedDict, Annotated, Literal
from langgraph.graph import StateGraph, START, END
from sklearn.ensemble import RandomForestRegressor
import json
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field

In [0]:
# --- 1. Define the Graph State ---
class WorkflowState(TypedDict):
    """
    Represents the state of our graph, tracking decisions made at each step.
    """
    user_query: str
    routing_decision_1: str # Changed to str to accommodate parsed LLM output
    routing_decision_2: str
    ml_prediction_value: float
    routing_decision_4: str
    routing_decision_5: str
    final_status_message: str

# --- 2. Create Dummy Dataset and Train a Simple ML Regressor (Same as before) ---

def train_dummy_regressor():
    np.random.seed(42)
    data = {
        'feature_age': np.random.randint(18, 65, 100),
        'feature_income': np.random.randint(30000, 100000, 100),
        'target_loan_amount': np.random.randint(5000, 50000, 100) + np.random.rand(100) * 1000
    }
    df = pd.DataFrame(data)
    X = df[['feature_age', 'feature_income']]
    y = df['target_loan_amount']
    model = RandomForestRegressor(n_estimators=10, random_state=42)
    model.fit(X, y)
    print("Dummy Random Forest Regressor Trained.")
    return model

ML_REGRESSOR = train_dummy_regressor()

# --- 3. Define Pydantic Schemas for LLM Output Parsing ---

class YesNoDecision(BaseModel):
    decision: Literal["Yes", "No"] = Field(description="Decides if the input is a valid business request.")

class AwaitingBType(BaseModel):
    decision: Literal["A", "B"] = Field(description="Classifies the type of request.")

class PriorityDecision(BaseModel):
    decision: Literal["Fast_Track", "Audit_Queue"] = Field(description="Determines processing priority.")

class FinalApproval(BaseModel):
    decision: Literal["Approved", "Denied"] = Field(description="The final approval decision.")

# --- 4. Define the Nodes (Processing Steps with actual LLM calls) ---

# Node 1 (LLM Binary Routing - Yes/No)
def node_1_llm_binary_router(state: WorkflowState) -> dict:
    """Uses an LLM to determine if the query is a valid business request (Yes/No)."""
    print(f"\n--- Node 1: LLM Routing (Yes/No) ---")
    prompt = ChatPromptTemplate.from_messages([
        ("system", "Analyze the user query. Is this related to finance, loans, or banking? Respond only with 'Yes' or 'No'."),
        ("human", "{query}")
    ])
    llm = ChatDatabricks(endpoint="databricks-meta-llama-3-3-70b-instruct", max_tokens=50)

    runnable = prompt | llm.with_structured_output(YesNoDecision)
    
    response = runnable.invoke({"query": state['user_query']})
    decision = response.decision
    print(f"LLM Decision: {decision}")
    return {"routing_decision_1": decision, "final_status_message": f"Initial Decision: {decision}"}

# Node 2 (LLM Binary Routing - A/B)
def node_2_llm_ab_router(state: WorkflowState) -> dict:
    """Uses an LLM to classify request type (Type A/Type B)."""
    print(f"--- Node 2: LLM Routing (A/B) ---")
    prompt = ChatPromptTemplate.from_messages([
        ("system", "Classify the request as Type A (Loan Application) or Type B (General Inquiry). Respond only with 'A' or 'B'."),
        ("human", "{query}")
    ])
    llm = ChatDatabricks(endpoint="databricks-meta-llama-3-3-70b-instruct", max_tokens=50)
    runnable = prompt | llm.with_structured_output(AwaitingBType)
    
    response = runnable.invoke({"query": state['user_query']})
    decision = response.decision
    print(f"LLM Decision: {decision}")
    return {"routing_decision_2": decision, "final_status_message": state["final_status_message"] + f" | Type: {decision}"}

# Node 3 (ML Regressor Routing - Non-LLM)
def node_3_ml_regressor_router(state: WorkflowState) -> dict:
    # ... [Same implementation as before, non-LLM node] ...
    print(f"--- Node 3: ML Regressor Prediction ---")
    input_data = pd.DataFrame([[random.randint(20, 60), random.randint(30000, 90000)]], 
                              columns=['feature_age', 'feature_income'])
    predicted_value = ML_REGRESSOR.predict(input_data)[0]
    print(f"Predicted Value: ${predicted_value:.2f}")
    
    return {"ml_prediction_value": predicted_value, 
            "final_status_message": state["final_status_message"] + f" | Value: ${predicted_value:.0f}"}

# Node 4 (LLM Binary Routing - Fast Track/Audit Queue)
def node_4_llm_priority_router(state: WorkflowState) -> dict:
    """Uses an LLM to assess priority and route to Fast Track or Audit."""
    print(f"--- Node 4: LLM Priority Routing ---")
    # This prompt uses previous decisions to inform the priority decision
    prompt = ChatPromptTemplate.from_messages([
        ("system", "Given the user query and prior steps, determine if this needs 'Fast_Track' or an 'Audit_Queue' review. Respond only with 'Fast_Track' or 'Audit_Queue'. Query details: {query}, Type: {prev_type}"),
        ("human", "Make a priority assessment.")
    ])
    llm = ChatDatabricks(endpoint="databricks-meta-llama-3-3-70b-instruct", max_tokens=50)

    runnable = prompt | llm.with_structured_output(PriorityDecision)
    response = runnable.invoke({"query": state['user_query'], "prev_type": state['routing_decision_2']})

    decision = response.decision
    print(f"LLM Decision: {decision}")
    return {"routing_decision_4": decision, "final_status_message": state["final_status_message"] + f" | Priority: {decision}"}

# Node 5 (LLM Binary Routing - Approved/Denied)
def node_5_llm_final_decision_router(state: WorkflowState) -> dict:
    """Uses an LLM for final review for approval or denial."""
    print(f"--- Node 5: LLM Final Decision Routing ---")
    # Final prompt uses all accumulated state data
    prompt = ChatPromptTemplate.from_messages([
        ("system", "Review the full process summary and make a final 'Approved' or 'Denied' decision. Respond only with 'Approved' or 'Denied'. Summary: {summary}"),
        ("human", "Final review.")
    ])
    llm = ChatDatabricks(endpoint="databricks-meta-llama-3-3-70b-instruct", max_tokens=50)

    runnable = prompt | llm.with_structured_output(FinalApproval)
    response = runnable.invoke({"summary": state['final_status_message']})
    
    decision = response.decision
    print(f"LLM Decision: {decision}")
    return {"routing_decision_5": decision, 
            "final_status_message": state["final_status_message"] + f" | Final: {decision}"}

# --- 5. Define the Conditional Edge Logic (Routers) ---
# These functions remain the same as the previous example, reading from the state dictionary.

def route_n1(state: WorkflowState) -> str:
    return state['routing_decision_1'] # "Yes" or "No"

def route_n2(state: WorkflowState) -> str:
    return state['routing_decision_2'] # "A" or "B"

def route_n3(state: WorkflowState) -> str:
    # Routes based on ML Regressor prediction value threshold
    if state['ml_prediction_value'] > 25000:
        return "HighValue"
    else:
        return "LowValue"

def route_n4(state: WorkflowState) -> str:
    return state['routing_decision_4'] # "Fast_Track" or "Audit_Queue"

def route_n5(state: WorkflowState) -> str:
    return state['routing_decision_5'] # "Approved" or "Denied"


# --- 6. Build the Graph (Same as before) ---
workflow = StateGraph(WorkflowState)

workflow.add_node("node_1", node_1_llm_binary_router)
workflow.add_node("node_2", node_2_llm_ab_router)
workflow.add_node("node_3", node_3_ml_regressor_router)
workflow.add_node("node_4", node_4_llm_priority_router)
workflow.add_node("node_5", node_5_llm_final_decision_router)

def final_approved_node(state): return {"final_status_message": state["final_status_message"] + " | Process Complete."}
def final_denied_node(state): return {"final_status_message": state["final_status_message"] + " | Terminated."}
def initial_rejection_node(state): return {"final_status_message": "Rejected immediately at step 1. Process Complete."}
workflow.add_node("final_approved", final_approved_node)
workflow.add_node("final_denied", final_denied_node)
workflow.add_node("initial_rejection", initial_rejection_node)

workflow.add_edge(START, "node_1")
workflow.add_conditional_edges("node_1", route_n1, {"Yes": "node_2", "No": "initial_rejection"})
workflow.add_edge("initial_rejection", END)
workflow.add_conditional_edges("node_2", route_n2, {"A": "node_3", "B": "node_3"}) 
workflow.add_conditional_edges("node_3", route_n3, {"HighValue": "node_4", "LowValue": "node_4"})
workflow.add_conditional_edges("node_4", route_n4, {"Fast_Track": "node_5", "Audit_Queue": "node_5"})
workflow.add_conditional_edges("node_5", route_n5, {"Approved": "final_approved", "Denied": "final_denied"})
workflow.add_edge("final_approved", END)
workflow.add_edge("final_denied", END)

app = workflow.compile()

# --- 7. Run Examples ---

print("--- Running Example 1: Full Path Simulation (Actual LLM Calls) ---")
# Example input: A standard loan application request
input_state = {"user_query": "I need a loan application processed ASAP for home equity."}

# Running the app will now make 4 distinct API calls to the LLMs
events = app.stream(input_state, stream_mode="updates")
for event in events:
    if "__end__" in event:
        print("Workflow Ended.")

final_state_1 = app.invoke(input_state)
print(f"\nFinal Status 1: {final_state_1['final_status_message']}")

print("\n" + "="*40 + "\n")

# print("--- Running Example 2: Early Termination Simulation (Node 1 'No') ---")
# # Example input: A query that should be rejected immediately by the LLM
# input_state_2 = {"user_query": "How do I fix my broken television screen?"}

# final_state_2 = app.invoke(input_state_2)
# print(f"\nFinal Status 2: {final_state_2['final_status_message']}")
